# 01 Prepare Synthetic Dataset With PaddleOCR

This notebook runs only data loading, translation, PaddleOCR, and synthetic image generation.

## 1. Environment

In [ ]:
%pip -q uninstall -y pillow
%pip -q install "pillow==11.3.0" "PyYAML==6.0.2"
%pip -q install -U --upgrade-strategy only-if-needed \
    "datasets==2.21.0" \
    "transformers==4.45.2" \
    "sentencepiece" "sacremoses" "tqdm"

%pip -q uninstall -y easyocr paddleocr paddlex paddlepaddle paddlepaddle-gpu opencv-python opencv-contrib-python opencv-python-headless scipy scikit-image
%pip -q install -U pip setuptools wheel
%pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4"
%pip -q install --no-deps --force-reinstall --no-cache-dir --prefer-binary "opencv-python-headless==4.10.0.84"
%pip -q install --no-deps --force-reinstall --no-cache-dir --prefer-binary "scipy==1.14.1" "scikit-image==0.25.2"
%pip -q install --prefer-binary "paddlepaddle==2.6.2"
%pip -q install --prefer-binary pyclipper lmdb rapidfuzz premailer attrdict visualdl fire
%pip -q install --no-deps --prefer-binary --only-binary=:all: "paddleocr==2.7.3"
%pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4"


In [ ]:
import sys
from pathlib import Path

module_candidates = [
    Path.cwd(),
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/kysyandrewik/prepare-synthetic-dataset'),
    Path('/kaggle/input/prepare-synthetic-dataset'),
    Path('/kaggle/input/datasets/kysyandrewik/rumodernvbert'),
    Path('/kaggle/input/rumodernvbert'),
]
module_candidates.extend(Path('/kaggle/input').glob('**/modernvbert_ru_prepare_paddleocr.py'))

MODULE_DIR = None
for candidate in module_candidates:
    candidate_dir = candidate.parent if candidate.name == 'modernvbert_ru_prepare_paddleocr.py' else candidate
    if (candidate_dir / 'modernvbert_ru_prepare_paddleocr.py').exists():
        MODULE_DIR = candidate_dir
        break
if MODULE_DIR is None:
    raise FileNotFoundError('Could not find modernvbert_ru_prepare_paddleocr.py in Kaggle working/input paths.')
sys.path.insert(0, str(MODULE_DIR))

import modernvbert_ru_prepare_paddleocr as prep

WORKDIR = prep.default_workdir('modernvbert_ru_ocr_artifact')
print('WORKDIR:', WORKDIR)
print('module:', prep.__file__)
print('default retriever model constant:', prep.DEFAULT_RETRIEVAL_MODEL_NAME)


## 2. Configuration

In [ ]:
EXPERIMENT_CONFIG = {
    'output_dir': WORKDIR,
    'max_rows': 300,
    'synthetic_subset_size': {'train': 100, 'val': 50, 'test': 100},
    'translation_model': 'Helsinki-NLP/opus-mt-en-ru',
    'font_path': None,
    'seed': 42,
    'streaming': True,
    'translator_device': 'cpu',
    'train_ratio': 0.8,
    'val_ratio': 0.1,
    'test_ratio': 0.1,
    'synthetic_splits': ('train', 'val', 'test'),
    'ocr_backend': 'paddleocr',
    'ocr_langs': ('en',),
    'ocr_use_angle_cls': True,
    'preprocess_for_ocr': True,
    'strict_ocr': True,
}

SMOKE_CONFIG = dict(EXPERIMENT_CONFIG)
SMOKE_CONFIG.update({
    'output_dir': WORKDIR / 'smoke',
    'max_rows': 12,
    'synthetic_subset_size': {'train': 4, 'val': 2, 'test': 2},
})

RUN_SMOKE = True
RUN_FULL_PREPARE = True


## 3. Full Artifact Preparation

In [ ]:
summary = None
if RUN_FULL_PREPARE:
    summary = prep.prepare_thesis_experiment(**EXPERIMENT_CONFIG)
    prep.export_json(WORKDIR / 'artifact_ready_summary.json', summary)
summary
